# K12 · DreamBooth-LoRA baseline on Stable Diffusion 1.5 (step 5)

One LoRA per image (rank 8, lr 1e-4, **400 steps** (reduced from 600 to finish in < 2 h), batch 1, 512 px, fp16, 8-bit Adam, no prior preservation) for the
8 images in `data/manifests/lora8.csv` (3 natural + 5 OOD), **two trainings at a time (one per T4)**. Then **15** samples
per image (30 DDIM steps, CFG 7.5, seeds 10..39), resized back to the input size.

Progress: one bar per image while training (the diffusers step counter), then one bar per image while sampling.
Limitation (documented): no dev-set tuning of steps / lr on the free tier; the diffusers example's defaults.

**How to run (Kaggle):**
1. Right panel → *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
2. Top right **Save Version** → *Save & Run All (Commit)* → Save. It runs in the background (you can close the tab);
   watch it under *View Active Events* / the version's **Logs** (progress lines every 2 min).
3. When it finishes, the result is the version's **Output**: `cs787_k12.zip`.
4. **Not finished?** (it stops itself before 2 hours) → *Add Input* → *Your Work* → this notebook's
   latest version → then *Save Version* again. It continues where it stopped.

In [ ]:
import time
T_START = time.time()                      # session clock: work stops before SESSION_HOURS
NOTEBOOK = "K12_lora"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
SESSION_HOURS = 1.9                        # hard stop: everything (incl. zipping) ends before 2 h

STEPS     = 400
N_SAMPLES = 15
SEED0     = 10
DIFFUSERS_VERSION = "0.35.1"
MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"

In [ ]:
import os, sys, subprocess, shutil, glob, json
if not os.path.isdir("/kaggle/working"):
    raise SystemExit("This notebook is written for Kaggle (kaggle.com/code).")
WORK, TMP = "/kaggle/working", "/tmp/cs787"
PROJECT = f"{TMP}/project"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(TMP, exist_ok=True)
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, PROJECT],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit("Could not download the code from GitHub. Turn Internet ON: right panel -> Session options -> "
                     "Internet (needs a phone-verified Kaggle account), then run again.\n" + r.stderr)
sys.path.insert(0, f"{PROJECT}/src")
os.environ["PYTHONPATH"] = f"{PROJECT}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")
from sidx.kaggle import gather_inputs, unpack_own, pack
from sidx.progress import Bars
print("code:", subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
gather_inputs()          # results of earlier versions attached under /kaggle/input (resume)

In [ ]:
import torch
N_GPUS = torch.cuda.device_count()
if N_GPUS == 0:
    raise SystemExit("No GPU. Right panel -> Session options -> Accelerator -> 'GPU T4 x2', then run again.")
cap = torch.cuda.get_device_capability(0)
if f"sm_{cap[0]}{cap[1]}" not in torch.cuda.get_arch_list():
    raise SystemExit(f"{torch.cuda.get_device_name(0)} is not supported by this PyTorch build: choose 'GPU T4 x2' (not P100).")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", N_GPUS, "GPU(s) -> all of them are used")

In [ ]:
# only the official repo's data downloader is needed here (no install: keeps K12's diffusers stack untouched)
SID_ROOT = f"{TMP}/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)
subprocess.run(["git", "-C", SID_ROOT, "checkout", "-q", SID_COMMIT], check=True)

In [ ]:
# natural images (balloons, forest, ...) are third-party and not in the repo: fetch + check sha256 against the manifests
r = subprocess.run([sys.executable, "scripts/prepare_natural.py", "--verify", "--sets", "uncond/sinddm15", "--sid-root", SID_ROOT],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
assert r.returncode == 0, "data check failed (see above)"

In [ ]:
!pip uninstall -q -y torchao
!pip install -q diffusers=={DIFFUSERS_VERSION} transformers==4.53.2 accelerate==1.9.0 peft==0.17.1 bitsandbytes pandas
!wget -q -O {TMP}/train_dreambooth_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v{DIFFUSERS_VERSION}/examples/dreambooth/train_dreambooth_lora.py
from huggingface_hub import snapshot_download      # download once, before two trainings start at the same time
snapshot_download(MODEL, allow_patterns=["*.json", "*.txt", "tokenizer/*", "text_encoder/model.safetensors",
                                         "unet/diffusion_pytorch_model.safetensors", "vae/diffusion_pytorch_model.safetensors"])
print("SD 1.5 cached")

## Train one LoRA per image, two at a time (skips images already trained)

In [ ]:
import csv
from sidx.gpupool import Task, run_pool, _safe
from sidx.progress import last_fraction
unpack_own()
rows = list(csv.DictReader(open(f"{PROJECT}/data/manifests/lora8.csv", encoding="utf-8")))
LORA = f"{WORK}/checkpoints/lora"
LOGS = f"{WORK}/logs/{NOTEBOOK}"
abs_path = lambda r: os.path.normpath(os.path.join(f"{PROJECT}/data/manifests", r["path"]))
weights_ready = lambda i: bool(glob.glob(f"{LORA}/{i}/pytorch_lora_weights.*"))

tasks = []
for r in rows:
    i = r["id"]
    if weights_ready(i):
        continue
    inst = f"{TMP}/lora_inst/{i}"; os.makedirs(inst, exist_ok=True)
    shutil.copy(abs_path(r), f"{inst}/img.png")
    tasks.append(Task(f"train_{i}", [sys.executable, f"{TMP}/train_dreambooth_lora.py", f"--pretrained_model_name_or_path={MODEL}",
        f"--instance_data_dir={inst}", f"--instance_prompt={r['prompt']}", f"--output_dir={TMP}/lora_out/{i}", "--resolution=512",
        "--train_batch_size=1", "--gradient_accumulation_steps=1", "--learning_rate=1e-4", "--lr_scheduler=constant",
        "--lr_warmup_steps=0", f"--max_train_steps={STEPS}", "--rank=8", "--mixed_precision=fp16", "--gradient_checkpointing",
        "--use_8bit_adam", "--checkpointing_steps=100000", "--seed=0"]))

def poll_train():
    res = {}
    for r in rows:
        i = r["id"]
        if weights_ready(i) or glob.glob(f"{TMP}/lora_out/{i}/pytorch_lora_weights.*"):
            res[i] = (STEPS, STEPS)
        else:
            f = last_fraction(f"{LOGS}/{_safe('train_' + i)}.log")
            res[i] = (min(f[0], STEPS), STEPS) if f else (0, STEPS)
    return res

print(len(tasks), "LoRAs to train")
if tasks:
    run_pool(tasks, poll_train, n_gpus=N_GPUS, log_dir=LOGS, unit="step", deadline=T_START + SESSION_HOURS * 3600 - 35 * 60)
for r in rows:                                  # keep the weights in the output
    i = r["id"]
    for f in glob.glob(f"{TMP}/lora_out/{i}/pytorch_lora_weights.*"):
        os.makedirs(f"{LORA}/{i}", exist_ok=True); shutil.copy(f, f"{LORA}/{i}/")
print({r["id"]: weights_ready(r["id"]) for r in rows})

## Sample 30 images per LoRA (live bar per image)

In [ ]:
import pandas as pd
from diffusers import StableDiffusionPipeline
from PIL import Image
OUT = f"{WORK}/outputs/phase8_lora/lora_r8_s{STEPS}__lora"
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{OUT}/job.json"):
    json.dump({"experiment": "phase8_lora", "job": f"lora_r8_s{STEPS}", "method": "lora_sd15",
               "params": {"rank": 8, "lr": 1e-4, "steps": STEPS, "resolution": 512, "sampler_steps": 30, "cfg": 7.5},
               "images": [{"id": r["id"], "path": abs_path(r)} for r in rows]}, open(f"{OUT}/job.json", "w"), indent=2)
summary = json.load(open(f"{LOGS}/_summary.json")) if os.path.exists(f"{LOGS}/_summary.json") else {}
cost = {r["id"]: {"train_s": summary.get(f"train_{r['id']}", {}).get("seconds")} for r in rows}
pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, safety_checker=None).to("cuda")
pipe.set_progress_bar_config(disable=True)
bars = Bars(unit="image", every_s=120)
try:
    for r in rows:
        i = r["id"]
        d = f"{OUT}/{i}"; os.makedirs(d, exist_ok=True)
        have = lambda: sum(os.path.exists(f"{d}/seed_{s:04d}.png") for s in range(SEED0, SEED0 + N_SAMPLES))
        bars.set(i, have(), N_SAMPLES)
        todo = [s for s in range(SEED0, SEED0 + N_SAMPLES) if not os.path.exists(f"{d}/seed_{s:04d}.png")]
        if not todo or not weights_ready(i) or time.time() > T_START + SESSION_HOURS * 3600 - 6 * 60:
            continue
        W, H = Image.open(abs_path(r)).size
        pipe.load_lora_weights(f"{LORA}/{i}")
        torch.cuda.reset_peak_memory_stats(); t0 = time.time()
        for s in todo:
            im = pipe(r["prompt"], num_inference_steps=30, guidance_scale=7.5, generator=torch.Generator("cuda").manual_seed(s)).images[0]
            im.resize((W, H), Image.LANCZOS).save(f"{d}/seed_{s:04d}.png")
            bars.set(i, have(), N_SAMPLES); bars.maybe_print()
        cost[i].update(sample_s_per_img=round((time.time() - t0) / len(todo), 2),
                       sample_peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1024**3, 2))
        pipe.unload_lora_weights()
finally:
    bars.close()
    os.makedirs(f"{WORK}/tables", exist_ok=True)
    json.dump(cost, open(f"{WORK}/tables/phase8_lora_cost.json", "w"), indent=2)
    pack("k12", kinds=("outputs", "checkpoints", "tables"))
pd.DataFrame(cost).T

In [ ]:
os.makedirs(f"{WORK}/logs", exist_ok=True)
!pip freeze > {WORK}/logs/{NOTEBOOK}_pip_freeze.txt
print("finished; output files:", sorted(os.listdir(WORK)))